In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import optuna

# ====================== FEATURE ENGINEERING ======================
def add_features(data):
    data = data.copy()
    data['Sample Date'] = pd.to_datetime(data['Sample Date'], format='%d-%m-%Y', errors='coerce')
    data = data.dropna(subset=['Sample Date'])
    
    data['lat'] = data['Latitude']
    data['lon'] = data['Longitude']
    
    # Trigonometric encoding for lat/lon
    for angle, prefix in [(data['lat'], 'lat'), (data['lon'], 'lon')]:
        rad = np.radians(angle)
        for k in [1, 2, 3, 4, 5]:
            data[f'{prefix}_sin_{k}'] = np.sin(k * rad)
            data[f'{prefix}_cos_{k}'] = np.cos(k * rad)
    
    # Time features
    data['year'] = data['Sample Date'].dt.year.astype(float)
    data['year_norm'] = (data['year'] - data['year'].min()) / (data['year'].max() - data['year'].min() + 1e-8)
    data['month'] = data['Sample Date'].dt.month.astype(float)
    data['doy']   = data['Sample Date'].dt.dayofyear.astype(float)
    
    # Cyclic encodings
    data['doy_sin_365'] = np.sin(2 * np.pi * data['doy'] / 365.25)
    data['doy_cos_365'] = np.cos(2 * np.pi * data['doy'] / 365.25)
    data['month_sin_12'] = np.sin(2 * np.pi * data['month'] / 12)
    data['month_cos_12'] = np.cos(2 * np.pi * data['month'] / 12)
    
    for period, label in [(182.625, '182_6'), (91.3125, '91_3'), (30.4375, '30_4'), (7.3, 'weekly')]:
        data[f'doy_sin_{label}'] = np.sin(2 * np.pi * data['doy'] / period)
        data[f'doy_cos_{label}'] = np.cos(2 * np.pi * data['doy'] / period)
    
    # Interaction features
    data['lat_doy_sin'] = data['lat_sin_1'] * data['doy_sin_365']
    data['lon_doy_sin'] = data['lon_sin_1'] * data['doy_sin_365']
    data['lat_year']    = data['lat'] * data['year_norm']
    data['lon_year']    = data['lon'] * data['year_norm']
    data['pet_lat']     = data.get('pet', 0) * data['lat']
    data['pet_doy_sin'] = data.get('pet', 0) * data['doy_sin_365']
    
    # Landsat band ratios
    epsilon = 1e-8
    data['turbidity_index'] = data.get('nir', 0) / (data.get('green', 0) + epsilon)
    data['phosphorus_index'] = (data.get('swir16', 0) - data.get('nir', 0)) / (data.get('swir16', 0) + data.get('nir', 0) + epsilon)
    data['ec_index'] = data.get('swir22', 0) / (data.get('swir16', 0) + epsilon)
    data['alkalinity_index'] = data.get('green', 0) / (data.get('swir22', 0) + epsilon)
    data['green_nir'] = data.get('green', 0) / (data.get('nir', 0) + epsilon)
    data['swir16_green'] = data.get('swir16', 0) / (data.get('green', 0) + epsilon)
    data['pet_ndmi'] = data.get('pet', 0) * data.get('NDMI', 0)
    data['pet_nir'] = data.get('pet', 0) * data.get('nir', 0)
    data['ndmi_swir22'] = data.get('NDMI', 0) * data.get('swir22', 0)
    
    return data

# ====================== LOAD DATA ======================
df_target = pd.read_csv('water_quality_training_dataset.csv')
df_terra = pd.read_csv('terraclimate_features_training.csv')
df_landsat = pd.read_csv('landsat_features_training.csv')
df_deltares = pd.read_csv('deltares_water_2010_2015_200_samples.csv')

# Load large extra Landsat datasets
df_landsat_extra1 = pd.read_csv('landsat_features_2000_2006_8000_samples.csv')
df_landsat_extra2 = pd.read_csv('landsat_features_2015_2023_8000.csv')

# Merge train
df_train = df_target.merge(df_terra, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = df_train.merge(df_landsat, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train = df_train.merge(df_deltares, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
df_train['is_train'] = 1

# Load sub
sub = pd.read_csv('submission_template.csv')
df_terra_val = pd.read_csv('terraclimate_features_validation.csv')
df_landsat_val = pd.read_csv('landsat_features_validation.csv')

sub = sub.merge(df_terra_val, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
sub = sub.merge(df_landsat_val, on=['Latitude', 'Longitude', 'Sample Date'], how='left')
sub['is_train'] = 0
sub['Total Alkalinity'] = np.nan
sub['Electrical Conductance'] = np.nan
sub['Dissolved Reactive Phosphorus'] = np.nan

# Prepare extras for concatenation to df_all (to enrich lags)
extras = [df_landsat_extra1, df_landsat_extra2, df_deltares]  # Include deltares full for extra data points
df_extras = pd.concat(extras, ignore_index=True)
df_extras = add_features(df_extras)

# Add missing columns to extras
missing_cols = set(df_train.columns) - set(df_extras.columns)
for col in missing_cols:
    df_extras[col] = np.nan if col in ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'] else 0
df_extras['is_train'] = 0

# Combine train and sub
df_all = pd.concat([df_train, sub], ignore_index=True)
df_all = add_features(df_all)

# Concat extras to df_all
df_all = pd.concat([df_all, df_extras], ignore_index=True)

# Remove duplicates if any
df_all = df_all.drop_duplicates(subset=['Latitude', 'Longitude', 'Sample Date'])

# Add lags per location
df_all = df_all.sort_values(['Latitude', 'Longitude', 'Sample Date'])
lag_features = ['pet', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'turbidity_index', 'phosphorus_index', 'ec_index', 'P', 'ETa', 'PET', 'Temp']
group_key = ['Latitude', 'Longitude']

for lag in [1, 2, 3, 7]:
    for f in lag_features:
        shift_series = df_all.groupby(group_key)[f].shift(lag)
        df_all[f'{f}_lag{lag}'] = shift_series.values
        df_all[f'{f}_delta{lag}'] = (df_all[f] - shift_series).values
        df_all[f'{f}_ratio{lag}'] = (df_all[f] / (shift_series + 1e-8)).values

# Rolling means
for win in [3, 7]:
    for f in lag_features:
        roll_mean = df_all.groupby(group_key)[f].rolling(win, min_periods=1).mean().reset_index(0, drop=True)
        df_all[f'{f}_roll{win}'] = roll_mean.values

df_all = df_all.fillna(0)

# Update features
lag_feats = [c for c in df_all.columns if '_lag' in c or '_delta' in c or '_ratio' in c or '_roll' in c]
raw_features = ['pet', 'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
                'P', 'ETa', 'PET', 'Melt', 'Snow', 'Temp', 'P_res', 'S_res', 
                'Ea_res', 'Qin_res', 'FracFull', 'Qout_res', 
                'turbidity_index', 'phosphorus_index', 'ec_index', 'alkalinity_index', 
                'green_nir', 'swir16_green', 'pet_ndmi', 'pet_nir', 'ndmi_swir22'] + lag_feats

engineered = [c for c in df_all.columns if any(x in c for x in [
    'sin', 'cos', 'year', 'doy_', 'month_', 'lat_', 'lon_', 'pet_lat', 'pet_doy_sin'
])]

features = raw_features + engineered

# Prepare train
df_train = df_all[df_all['is_train'] == 1].copy()
X = df_train[features]
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

targets = {
    'TA':  np.log1p(df_train['Total Alkalinity']),
    'EC':  np.log1p(df_train['Electrical Conductance']),
    'DRP': np.log1p(df_train['Dissolved Reactive Phosphorus'])
}

# ====================== HYPERPARAMETER TUNING WITH OPTUNA ======================
def objective(trial, X_scaled, y, model_type):
    if model_type == 'xgb':
        params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.02),
            'max_depth': trial.suggest_int('max_depth', 4, 9),
            'subsample': trial.suggest_float('subsample', 0.6, 0.95),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
            'tree_method': 'hist',
            'seed': 42
        }
        num_boost_round = 5000
    else:  # lgb
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.02),
            'max_depth': trial.suggest_int('max_depth', 4, 9),
            'num_leaves': trial.suggest_int('num_leaves', 15, 255),
            'subsample': trial.suggest_float('subsample', 0.6, 0.95),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
            'min_child_weight': trial.suggest_float('min_child_weight', 0.001, 0.2),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
            'n_jobs': -1,
            'verbosity': -1,
            'random_state': 42,
            'force_col_wise': True
        }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    
    for tr_idx, val_idx in kf.split(X_scaled):
        X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        if model_type == 'xgb':
            dtrain = xgb.DMatrix(X_tr, label=y_tr)
            dval = xgb.DMatrix(X_val, label=y_val)
            booster = xgb.train(
                params,
                dtrain,
                num_boost_round=5000,
                evals=[(dval, 'eval')],
                early_stopping_rounds=200,
                verbose_eval=False
            )
            pred = booster.predict(dval)
        else:
            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(200, verbose=False)]
            )
            pred = model.predict(X_val)
        
        oof[val_idx] = pred
    
    return np.sqrt(mean_squared_error(y, oof))

# Tune for each target and model
best_params = {}
for name, y in targets.items():
    print(f"\nTuning for {name}")
    
    # Tune XGBoost
    study_xgb = optuna.create_study(direction='minimize')
    study_xgb.optimize(lambda trial: objective(trial, X_scaled, y, 'xgb'), n_trials=50)
    best_params[f'{name}_xgb'] = study_xgb.best_params
    
    # Tune LightGBM
    study_lgb = optuna.create_study(direction='minimize')
    study_lgb.optimize(lambda trial: objective(trial, X_scaled, y, 'lgb'), n_trials=50)
    best_params[f'{name}_lgb'] = study_lgb.best_params

# ====================== TRAINING WITH BEST PARAMS ======================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {}

for name, y in targets.items():
    print(f"\n=== Training {name} with best params ===")
    oof = np.zeros(len(y))
    xgb_list = []
    lgb_list = []
    
    xgb_params = best_params[f'{name}_xgb']
    xgb_params.update({'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'seed': 42})
    
    lgb_params = best_params[f'{name}_lgb']
    lgb_params.update({'objective': 'regression', 'metric': 'rmse', 'n_jobs': -1, 'verbosity': -1, 'random_state': 42, 'force_col_wise': True})
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_scaled)):
        X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        # XGBoost
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        booster = xgb.train(
            xgb_params,
            dtrain,
            num_boost_round=5000,
            evals=[(dval, 'eval')],
            early_stopping_rounds=200,
            verbose_eval=200
        )
        xgb_list.append(booster)
        
        # LightGBM
        lgb_model = lgb.LGBMRegressor(**lgb_params)
        lgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(200)]
        )
        lgb_list.append(lgb_model)
        
        # OOF
        xgb_pred = booster.predict(dval)
        lgb_pred = lgb_model.predict(X_val)
        blend_pred = (xgb_pred * 0.5  + lgb_pred * 0.5)  # Equal blend
        oof[val_idx] = blend_pred
    
    orig_y = df_train[name.replace('TA', 'Total Alkalinity').replace('EC', 'Electrical Conductance').replace('DRP', 'Dissolved Reactive Phosphorus')]
    oof_back = np.expm1(oof)
    r2 = r2_score(orig_y, oof_back)
    rmse = np.sqrt(mean_squared_error(orig_y, oof_back))
    print(f"{name}  CV R² = {r2:.4f}   RMSE = {rmse:.2f}")
    
    models[name] = (xgb_list, lgb_list)

# ====================== SUBMISSION ======================
sub = df_all[df_all['is_train'] == 0].copy()

sub_X = sub[features]
sub_X_scaled = scaler.transform(sub_X)
dsub = xgb.DMatrix(sub_X_scaled)

def blend_predict(xgb_models, lgb_models, X, dX):
    xgb_preds = np.mean([m.predict(dX) for m in xgb_models], axis=0)
    lgb_preds = np.mean([m.predict(X) for m in lgb_models], axis=0)
    return (xgb_preds * 0.5 + lgb_preds * 0.5)

sub['Total Alkalinity'] = np.expm1(blend_predict(models['TA'][0], models['TA'][1], sub_X_scaled, dsub))
sub['Electrical Conductance'] = np.expm1(blend_predict(models['EC'][0], models['EC'][1], sub_X_scaled, dsub))
sub['Dissolved Reactive Phosphorus'] = np.expm1(blend_predict(models['DRP'][0], models['DRP'][1], sub_X_scaled, dsub))

# Clip
sub['Total Alkalinity'] = sub['Total Alkalinity'].clip(5, 400)
sub['Electrical Conductance'] = sub['Electrical Conductance'].clip(30, 2200)
sub['Dissolved Reactive Phosphorus'] = sub['Dissolved Reactive Phosphorus'].clip(0.5, 250)

final_sub = sub[['Latitude', 'Longitude', 'Sample Date',
                 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

final_sub.to_csv('submission_large_extras_lag_blend.csv', index=False)
print("✅ Saved → submission_large_extras_lag_blend.csv")
print("Upload this file and check the leaderboard!")

/var/folders/hv/kxrxx3qn2rz5xqvmhb82grsw0000gn/T/ipykernel_14945/907878858.py:108: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_all, df_extras], ignore_index=True)
/var/folders/hv/kxrxx3qn2rz5xqvmhb82grsw0000gn/T/ipykernel_14945/907878858.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all[f'{f}_delta{lag}'] = (df_all[f] - shift_series).values
/var/folders/hv/kxrxx3qn2rz5xqvmhb82grsw0000gn/T/ipykernel_14945/907878858.py:123: PerformanceWarning: DataFrame is highly fragmented.  Th


Tuning for TA


[I 2026-03-04 13:16:52,016] Trial 0 finished with value: 0.3218420048085034 and parameters: {'learning_rate': 0.0037725714439243957, 'max_depth': 5, 'subsample': 0.8832279671189017, 'colsample_bytree': 0.6369578713892177, 'min_child_weight': 6, 'reg_alpha': 1.6686313011892997, 'reg_lambda': 1.7099026643456081}. Best is trial 0 with value: 0.3218420048085034.
[I 2026-03-04 13:18:22,119] Trial 1 finished with value: 0.31035673839284855 and parameters: {'learning_rate': 0.006634505221692, 'max_depth': 7, 'subsample': 0.767416567044816, 'colsample_bytree': 0.7076234227595579, 'min_child_weight': 8, 'reg_alpha': 1.873253411739544, 'reg_lambda': 0.6892850766216958}. Best is trial 1 with value: 0.31035673839284855.
[I 2026-03-04 13:20:27,540] Trial 2 finished with value: 0.3055581932059859 and parameters: {'learning_rate': 0.005713661150357615, 'max_depth': 8, 'subsample': 0.7792295335630114, 'colsample_bytree': 0.7694428534017821, 'min_child_weight': 7, 'reg_alpha': 0.6000356970496499, 'reg_


Tuning for EC


[I 2026-03-04 14:31:38,773] Trial 0 finished with value: 0.2721758372848427 and parameters: {'learning_rate': 0.0019978230065035543, 'max_depth': 8, 'subsample': 0.6362214729837594, 'colsample_bytree': 0.6362663969578366, 'min_child_weight': 10, 'reg_alpha': 1.0566408181793108, 'reg_lambda': 1.771830127616704}. Best is trial 0 with value: 0.2721758372848427.
[I 2026-03-04 14:32:20,070] Trial 1 finished with value: 0.2793822161254201 and parameters: {'learning_rate': 0.00853905648150262, 'max_depth': 4, 'subsample': 0.9049124087086299, 'colsample_bytree': 0.8684513325242654, 'min_child_weight': 9, 'reg_alpha': 0.9590131108035451, 'reg_lambda': 1.6318394049600922}. Best is trial 0 with value: 0.2721758372848427.
[I 2026-03-04 14:33:22,900] Trial 2 finished with value: 0.27252210177816133 and parameters: {'learning_rate': 0.01758931254881017, 'max_depth': 6, 'subsample': 0.8889110116004187, 'colsample_bytree': 0.9392740398163606, 'min_child_weight': 4, 'reg_alpha': 1.984944667982704, 'reg

KeyboardInterrupt: 